# Lab 03 — Event Hubs Producer

Read live Wikimedia recent-change events and send them to Azure Event Hubs.

This notebook:

1. Loads the shared Lab 03 configuration.
2. Connects to the Wikimedia Server-Sent Events stream.
3. Enriches each event with producer metadata.
4. Sends events to Azure Event Hubs in batches.


## 1. Install the Azure Event Hubs client

Run this cell once on a new cluster. Databricks may ask you to restart Python after installation.


In [0]:
%pip install azure-eventhub


## 2. Load shared configuration


In [0]:
%run ./lab03_config


## 3. Producer settings

`max_events` limits the run so the notebook does not stream forever.  
Increase it later when testing the full pipeline.


In [0]:
import json
import time
from datetime import datetime, timezone

import requests
from azure.eventhub import EventData
from azure.eventhub import EventHubProducerClient


WIKIMEDIA_STREAM_URL = (
    "https://stream.wikimedia.org/v2/stream/recentchange"
)

max_events = 100
request_timeout_seconds = 90
batch_event_limit = 25

print(f"Wikimedia stream: {WIKIMEDIA_STREAM_URL}")
print(f"Event Hub: {eventhub_name}")
print(f"Producer ID: {producer_id}")
print(f"Maximum events: {max_events}")


## 4. Load the Event Hub connection string safely


In [0]:
connection_string = dbutils.secrets.get(
    scope=secret_scope,
    key=eventhub_secret_key
)

print("Event Hub secret loaded successfully.")


## 5. Create the Event Hub producer client

An entity-level connection string normally contains `EntityPath`.  
The code supports both entity-level and namespace-level connection strings.


In [0]:
if "EntityPath=" in connection_string:
    producer_client = EventHubProducerClient.from_connection_string(
        conn_str=connection_string
    )
else:
    producer_client = EventHubProducerClient.from_connection_string(
        conn_str=connection_string,
        eventhub_name=eventhub_name
    )

eventhub_properties = producer_client.get_eventhub_properties()

print("Connected to Azure Event Hubs.")
print(f"Resolved Event Hub: {eventhub_properties['eventhub_name']}")
print(f"Partitions: {eventhub_properties['partition_ids']}")


## 6. Wikimedia SSE parser and batch sender


In [0]:
def enrich_event(raw_event: dict) -> dict:
    """Add ingestion metadata without removing Wikimedia fields."""

    enriched = dict(raw_event)

    enriched["_producer_id"] = producer_id
    enriched["_source_system"] = "wikimedia_recentchange"
    enriched["_ingested_at_utc"] = datetime.now(
        timezone.utc
    ).isoformat()

    return enriched


def send_event_batch(
    producer: EventHubProducerClient,
    events: list[dict]
) -> int:
    """Send JSON events while respecting Event Hub batch-size limits."""

    if not events:
        return 0

    sent_count = 0
    event_batch = producer.create_batch()

    for event in events:
        event_data = EventData(
            json.dumps(
                event,
                ensure_ascii=False,
                separators=(",", ":")
            )
        )

        event_data.content_type = "application/json"
        event_data.properties = {
            "producer_id": producer_id,
            "source_system": "wikimedia_recentchange"
        }

        try:
            event_batch.add(event_data)

        except ValueError:
            if len(event_batch) == 0:
                raise ValueError(
                    "A single Wikimedia event is larger than the "
                    "maximum Event Hub batch size."
                )

            producer.send_batch(event_batch)
            sent_count += len(event_batch)

            event_batch = producer.create_batch()
            event_batch.add(event_data)

    if len(event_batch) > 0:
        producer.send_batch(event_batch)
        sent_count += len(event_batch)

    return sent_count


def stream_wikimedia_events(
    url: str,
    event_limit: int
):
    """Yield JSON objects from the Wikimedia SSE stream."""

    headers = {
        "Accept": "text/event-stream",
        "User-Agent": (
            "DatabricksAcademy-Lab03/"
            f"{producer_id}"
        )
    }

    with requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=(15, request_timeout_seconds)
    ) as response:

        response.raise_for_status()

        data_lines = []

        for raw_line in response.iter_lines(
            decode_unicode=True
        ):
            if raw_line is None:
                continue

            line = raw_line.strip()

            if not line:
                if data_lines:
                    payload = "\n".join(data_lines)
                    data_lines = []

                    try:
                        yield json.loads(payload)
                    except json.JSONDecodeError:
                        continue

                continue

            if line.startswith(":"):
                continue

            if line.startswith("data:"):
                data_lines.append(
                    line.removeprefix("data:").lstrip()
                )


## 7. Read live Wikimedia events and send them to Event Hubs


In [0]:
events_buffer = []
received_count = 0
sent_count = 0
started_at = time.time()

try:
    with producer_client:
        for wikimedia_event in stream_wikimedia_events(
            url=WIKIMEDIA_STREAM_URL,
            event_limit=max_events
        ):
            events_buffer.append(
                enrich_event(wikimedia_event)
            )
            received_count += 1

            if len(events_buffer) >= batch_event_limit:
                sent_count += send_event_batch(
                    producer=producer_client,
                    events=events_buffer
                )

                print(
                    f"Received: {received_count} | "
                    f"Sent: {sent_count}"
                )

                events_buffer.clear()

            if received_count >= max_events:
                break

        if events_buffer:
            sent_count += send_event_batch(
                producer=producer_client,
                events=events_buffer
            )
            events_buffer.clear()

except requests.RequestException as exc:
    raise RuntimeError(
        f"Wikimedia stream request failed: {exc}"
    ) from exc

elapsed_seconds = round(time.time() - started_at, 2)

print("Producer run completed.")
print(f"Events received: {received_count}")
print(f"Events sent: {sent_count}")
print(f"Elapsed seconds: {elapsed_seconds}")

if received_count != sent_count:
    raise RuntimeError(
        "Received and sent event counts do not match."
    )


## 8. Optional: display one sample event

This reconnects briefly to the live stream and displays one event without exposing the Event Hub secret.


In [0]:
sample_event = next(
    stream_wikimedia_events(
        url=WIKIMEDIA_STREAM_URL,
        event_limit=1
    )
)

sample_summary = {
    "wiki": sample_event.get("wiki"),
    "type": sample_event.get("type"),
    "title": sample_event.get("title"),
    "user": sample_event.get("user"),
    "bot": sample_event.get("bot"),
    "timestamp": sample_event.get("timestamp"),
}

display(spark.createDataFrame([sample_summary]))
